# 17. Python 多线程和多进程

多线程和多进程都用于并发执行任务。

本章重点内容：

- 并发、并行、线程、进程
- `threading.Thread`
- 线程参数和线程等待
- 线程锁 `Lock`
- 队列 `queue.Queue`
- 线程池 `ThreadPoolExecutor`
- GIL 的基本影响
- `multiprocessing`
- 进程池 `ProcessPoolExecutor`
- 多线程和多进程如何选择

注意：Windows 和 Jupyter Notebook 中运行多进程示例时，经常需要把代码放到 `.py` 文件，并写在 `if __name__ == "__main__":` 下面。


## 1. 基本概念

- 并发：多个任务交替推进，看起来像同时进行。
- 并行：多个任务真的同时运行，通常需要多核 CPU。
- 线程：进程内部的执行单元，共享同一个进程的内存。
- 进程：操作系统分配资源的基本单位，进程之间内存相互独立。

一般来说：

- I/O 密集型任务适合多线程，例如网络请求、文件读写。
- CPU 密集型任务适合多进程，例如大量计算、图像处理。


## 2. 创建线程

`threading.Thread` 可以创建线程。线程启动后，主程序和子线程可以同时推进。


In [1]:
import threading
from time import sleep


def download(file_name):
    print(f'开始下载：{file_name}')
    sleep(1)
    print(f'下载完成：{file_name}')


thread1 = threading.Thread(target=download, args=('a.txt',))
thread2 = threading.Thread(target=download, args=('b.txt',))

thread1.start()
thread2.start()

# join() 表示等待线程执行结束
thread1.join()
thread2.join()

print('所有下载任务完成')


开始下载：a.txt
开始下载：b.txt
下载完成：b.txt
下载完成：a.txt
所有下载任务完成


### 解释

- `target=download` 表示线程要执行的函数。
- `args=('a.txt',)` 是传给函数的位置参数，单个参数的元组要写逗号。
- `start()` 启动线程。
- `join()` 等待线程结束，避免主程序提前结束。


## 3. 继承 `Thread`

除了传入 `target`，也可以继承 `threading.Thread`，重写 `run()` 方法。


In [2]:
import threading
from time import sleep


class Worker(threading.Thread):
    def __init__(self, name, seconds):
        super().__init__()
        self.name = name
        self.seconds = seconds

    def run(self):
        print(f'{self.name} 开始工作')
        sleep(self.seconds)
        print(f'{self.name} 工作完成')


worker1 = Worker('线程 A', 1)
worker2 = Worker('线程 B', 1)

worker1.start()
worker2.start()

worker1.join()
worker2.join()


线程 A 开始工作
线程 B 开始工作
线程 A 工作完成
线程 B 工作完成


### 解释

- 继承 `Thread` 时，把线程要执行的逻辑写在 `run()` 中。
- 启动线程仍然调用 `start()`，不要直接调用 `run()`。
- 直接调用 `run()` 只是普通函数调用，不会创建新线程。


## 4. 线程共享数据和锁

多个线程会共享同一份内存。如果同时修改同一个变量，可能出现竞争问题。可以使用 `Lock` 保证同一时刻只有一个线程进入关键区域。


In [3]:
import threading

counter = 0
lock = threading.Lock()


def add_many_times():
    global counter

    for _ in range(100000):
        # with lock 可以自动获取和释放锁
        with lock:
            counter += 1


threads = [threading.Thread(target=add_many_times) for _ in range(5)]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print('counter =', counter)


counter = 500000


### 解释

- 多线程共享全局变量 `counter`。
- `counter += 1` 不是绝对安全的原子操作。
- `Lock` 可以保护关键区域，避免多个线程同时修改共享数据。
- 锁用得太多会降低并发效率，还可能造成死锁。


## 5. 使用队列在线程间传递数据

`queue.Queue` 是线程安全的队列，适合实现生产者-消费者模型。


In [4]:
import queue
import threading
from time import sleep

task_queue = queue.Queue()


def producer():
    for item in ['任务1', '任务2', '任务3']:
        print('生产：', item)
        task_queue.put(item)
        sleep(0.2)

    # 放入 None 作为结束信号
    task_queue.put(None)


def consumer():
    while True:
        task = task_queue.get()

        if task is None:
            print('消费者收到结束信号')
            break

        print('消费：', task)
        task_queue.task_done()


t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)

t1.start()
t2.start()

t1.join()
t2.join()


生产： 任务1
消费： 任务1
生产： 任务2
消费： 任务2
生产： 任务3
消费： 任务3
消费者收到结束信号


### 解释

- `put()` 往队列中放数据。
- `get()` 从队列中取数据。
- 队列内部已经处理了线程同步问题。
- 可以使用特殊值，例如 `None`，通知消费者结束。


## 6. 线程池

线程池可以复用线程，避免手动创建和管理大量线程。标准库中的 `ThreadPoolExecutor` 很常用。


In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep


def fetch(url):
    print('开始请求：', url)
    sleep(0.5)
    return f'{url} 请求完成'


urls = ['url1', 'url2', 'url3', 'url4']

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(fetch, url) for url in urls]

    for future in as_completed(futures):
        print(future.result())


开始请求： url1
开始请求： url2
开始请求： url3
url2 请求完成
开始请求： url4
url1 请求完成
url3 请求完成
url4 请求完成


### 解释

- `max_workers=2` 表示最多同时运行 2 个线程。
- `submit()` 提交任务，返回 `Future` 对象。
- `future.result()` 获取任务返回值，如果任务中抛出异常，这里也会重新抛出。
- `as_completed()` 会按任务完成顺序返回结果。


## 7. GIL 简要理解

CPython 中有 GIL，也就是全局解释器锁。它会让同一时刻只有一个线程执行 Python 字节码。

这意味着：

- 多线程适合 I/O 密集型任务，因为等待 I/O 时线程可以切换。
- 多线程不适合提升纯 Python CPU 密集型计算速度。
- CPU 密集型任务可以考虑多进程。


In [6]:
from time import perf_counter


def cpu_task(n):
    total = 0
    for i in range(n):
        total += i * i
    return total


start = perf_counter()
result = cpu_task(1_000_000)
end = perf_counter()

print(result)
print(f'单线程计算耗时：{end - start:.4f} 秒')


333332833333500000
单线程计算耗时：0.0299 秒


### 解释

- 上面的任务主要消耗 CPU。
- 对这类纯计算任务，多线程通常不会明显加速。
- 如果任务主要在等待网络、磁盘、数据库，多线程仍然很有价值。


## 8. 多进程基础

多进程可以绕开 GIL，让多个 Python 进程在多核 CPU 上并行执行。

在 Windows 和 Notebook 环境中，多进程代码建议写到 `.py` 文件里运行，并放在 `if __name__ == "__main__":` 保护下。


In [7]:
from pathlib import Path

RES_DIR = Path('res')
if not RES_DIR.exists() and Path('python_learn/res').exists():
    RES_DIR = Path('python_learn/res')

script_path = RES_DIR / 'chapter17_process_demo.py'

print('多进程示例脚本：', script_path.resolve())
print('可以在终端运行：')
print(f'python "{script_path}"')


多进程示例脚本： C:\Users\11435\Desktop\python_base\python_learn\res\chapter17_process_demo.py
可以在终端运行：
python "res\chapter17_process_demo.py"


### 示例脚本核心内容

```python
from concurrent.futures import ProcessPoolExecutor

def square(num):
    return num * num

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(square, [1, 2, 3, 4]))
    print(results)
```

### 解释

- 多进程会启动新的 Python 解释器进程。
- Windows 下必须用 `if __name__ == "__main__":` 避免子进程重复创建子进程。
- 传给进程池的函数通常要定义在模块顶层，不能是局部函数。


## 9. 进程池 `ProcessPoolExecutor`

`ProcessPoolExecutor` 和 `ThreadPoolExecutor` 用法类似，但底层使用多个进程。


In [8]:
# 下面是适合放到 .py 文件中运行的进程池示例。
# 在 Notebook 中直接运行时，Windows 环境可能会因为函数无法被子进程导入而失败。

from concurrent.futures import ProcessPoolExecutor


def square(num):
    return num * num


if __name__ == '__main__':
    numbers = [1, 2, 3, 4, 5]

    with ProcessPoolExecutor(max_workers=2) as executor:
        results = list(executor.map(square, numbers))

    print(results)


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

### 解释

- `ProcessPoolExecutor` 适合 CPU 密集型任务。
- 进程之间内存独立，传递数据需要序列化，所以小任务过多时开销可能很大。
- 如果任务很轻，进程池的创建和通信成本可能超过收益。


## 10. 多线程和多进程怎么选

| 场景 | 推荐方式 |
| --- | --- |
| 网络请求、文件读写、数据库访问 | 多线程 |
| 大量 CPU 计算、图像处理、数据压缩 | 多进程 |
| 任务数量很多但单个任务很轻 | 先考虑线程池或异步 |
| 需要共享大量内存状态 | 多线程更方便，但要注意锁 |
| 需要利用多核 CPU | 多进程 |

选型时不要只看“哪个更高级”，要看任务到底是在等 I/O，还是在吃 CPU。


## 11. 小练习：综合示例

1. 使用线程池模拟下载 5 个文件。
2. 使用锁保护共享计数器。
3. 查看 `res/chapter17_process_demo.py`，在终端运行它观察进程 ID。


In [9]:
from concurrent.futures import ThreadPoolExecutor
from time import sleep
import threading


def download_file(file_name):
    sleep(0.2)
    return f'{file_name} 下载完成'


files = [f'file_{i}.txt' for i in range(1, 6)]

with ThreadPoolExecutor(max_workers=3) as executor:
    for result in executor.map(download_file, files):
        print(result)


counter = 0
lock = threading.Lock()


def increase():
    global counter
    for _ in range(1000):
        with lock:
            counter += 1


threads = [threading.Thread(target=increase) for _ in range(3)]

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print('最终计数：', counter)


file_1.txt 下载完成
file_2.txt 下载完成
file_3.txt 下载完成
file_4.txt 下载完成
file_5.txt 下载完成
最终计数： 3000


## 12. 常见错误总结

1. 创建线程后忘记调用 `start()`。
2. 直接调用 `run()`，误以为启动了新线程。
3. 主线程没有 `join()`，导致还没等子线程完成就继续往下走。
4. 多线程修改共享变量时没有加锁。
5. 锁获取后没有释放，导致死锁；推荐使用 `with lock:`。
6. 把 CPU 密集型任务交给多线程，期望明显提速。
7. Windows 多进程代码没有写 `if __name__ == "__main__":`。
8. 把局部函数、lambda 传给进程池，导致无法序列化。
9. 进程之间误以为可以直接共享普通全局变量。
10. 任务太小却大量使用多进程，进程通信开销反而更大。
